<a href="https://colab.research.google.com/github/TnT1425/SearchMovies/blob/main/SM_ServerAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
# Cài đặt MongoDB vào Colab
!apt-get install -y mongodb > /dev/null 2>&1
!service mongodb start
!pip install nltk

# Cài đặt các thư viện AI và Server
!pip install fastapi uvicorn nest-asyncio easyocr transformers httpx pymongo pyngrok python-multipart torch torchvision torchaudio > /dev/null 2>&1
print("✅ Đã cài đặt xong toàn bộ môi trường!")

mongodb: unrecognized service
✅ Đã cài đặt xong toàn bộ môi trường!


In [14]:
import os
import io
import re
from fastapi import FastAPI, File, UploadFile
from datetime import datetime
import httpx
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration
import easyocr
import nest_asyncio
import uvicorn
import torch
from pyngrok import ngrok
import nltk
from nltk.corpus import wordnet
nltk.download('wordnet')

# 1. BẬT GPU ĐỂ TĂNG TỐC ĐỘ XỬ LÝ
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Đang chạy Server trên: {device.upper()}")

app = FastAPI()

# 2. API KEY VÀ "DATABASE" TRÊN RAM
TMDB_API_KEY = "06d01d3b9f5adbd2cecf647a6b1fa8b0"
TMDB_BASE_URL = "https://api.themoviedb.org/3"

# Dùng List lưu tạm thay cho MongoDB để tránh lỗi ngầm của Colab
search_history_db = []

# 3. NẠP AI VÀO GPU
print("Đang nạp OCR...")
reader = easyocr.Reader(['en'], gpu=(device=="cuda"))

print("Đang nạp BLIP...")
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)

print("✅ AI SẴN SÀNG!")


@app.get("/search-list")
async def search_movie_list(query: str):
    if len(query) < 2:
        return []

    async with httpx.AsyncClient() as client_http:
        # THAY ĐỔI 1: Dùng /search/multi thay vì /search/movie
        search_url = f"{TMDB_BASE_URL}/search/multi"
        params = {"api_key": TMDB_API_KEY, "query": query, "language": "vi-VN"}
        try:
            response = await client_http.get(search_url, params=params)
            data = response.json()

            results = []
            for item in data.get("results", [])[:10]:
                media_type = item.get("media_type")

                # Xử lý nếu kết quả là Phim hoặc TV Show
                if media_type in ["movie", "tv"]:
                    title = item.get("title") or item.get("name")
                    release_date = item.get("release_date") or item.get("first_air_date", "N/A")
                    poster_path = item.get("poster_path")

                    results.append({
                        "title": title,
                        "release_date": release_date,
                        "overview": item.get("overview", "Chưa có mô tả"),
                        "poster_path": f"https://image.tmdb.org/t/p/w200{poster_path}" if poster_path else None
                    })

                # Xử lý nếu kết quả là Diễn viên / Người nổi tiếng
                elif media_type == "person":
                    # Lấy danh sách phim nổi bật của người này
                    known_for = [k.get("title") or k.get("name") for k in item.get("known_for", [])]
                    known_for_str = ", ".join(filter(None, known_for))
                    profile_path = item.get("profile_path")

                    results.append({
                        "title": item.get("name"), # Tên diễn viên
                        "release_date": "Diễn viên / Đạo diễn",
                        "overview": f"Được biết đến qua các phim: {known_for_str}",
                        "poster_path": f"https://image.tmdb.org/t/p/w200{profile_path}" if profile_path else None
                    })

            return results
        except Exception as e:
            return []

@app.post("/identify-and-save")
async def identify_and_save(query_name: str):
    async with httpx.AsyncClient() as client_http:
        # Hỗ trợ tìm cả phim, tv show và diễn viên
        search_url = f"{TMDB_BASE_URL}/search/multi"

        # --- THUẬT TOÁN SEARCH ENGINE THÔNG MINH ---
        words = query_name.split()
        found_results = []

        # Vòng lặp: Nếu không tìm thấy, xóa bớt chữ cuối cùng và tìm lại
        while len(words) > 0:
            current_query = " ".join(words)
            print(f"🔍 Đang thử tìm TMDb với từ khóa: '{current_query}'")

            params = {"api_key": TMDB_API_KEY, "query": current_query, "language": "vi-VN"}
            response = await client_http.get(search_url, params=params)
            data = response.json()

            # Chỉ lấy các kết quả hợp lệ (phim, show, hoặc người)
            results = [item for item in data.get("results", []) if item.get("media_type") in ["movie", "tv", "person"]]

            if results:
                found_results = results
                print(f"✅ BÙM! Đã tìm thấy phim với từ khóa: '{current_query}'")
                break # Tìm thấy là thoát vòng lặp ngay

            # Nếu không tìm thấy, xóa bớt chữ cuối cùng rồi quay lại đầu vòng lặp
            words.pop()
        # ---------------------------------------------

        if not found_results:
            return {"message": "Không tìm thấy phim"}

        # Xử lý kết quả đầu tiên tìm được
        item = found_results[0]
        media_type = item.get("media_type")
        title = item.get("title") or item.get("name")
        release_date = item.get("release_date") or item.get("first_air_date", "N/A")

        # Nếu là người (diễn viên), lấy danh sách phim làm mô tả
        if media_type == "person":
            known_for = [k.get("title") or k.get("name") for k in item.get("known_for", [])]
            overview = "Diễn viên. Nổi tiếng qua: " + ", ".join(filter(None, known_for))
        else:
            overview = item.get("overview", "Chưa có mô tả")

        poster_path = item.get("poster_path") or item.get("profile_path")

        movie_info = {
            "_id": str(len(search_history_db) + 1),
            "tmdb_id": item["id"],
            "title": title,
            "release_date": release_date,
            "overview": overview,
            "poster_path": f"https://image.tmdb.org/t/p/w500{poster_path}" if poster_path else None,
            "search_time": datetime.utcnow().isoformat()
        }

        search_history_db.append(movie_info)
        return {"id": movie_info["_id"], "message": "Đã lưu", "movie_info": movie_info}

def extract_keywords(raw_text: str) -> str:
    # Thêm các từ rác do AI hay đẻ ra như: suit, costume, outfit, character, standing...
    stop_words = ["a", "an", "the", "in", "on", "of", "with", "is", "poster", "movie", "film",
                  "picture", "image", "featuring", "showing", "close up", "text", "background",
                  "presents", "imax", "3d", "cinema", "suit", "costume", "outfit", "character",
                  "standing", "sitting", "holding", "wearing", "sign", "cover"]

    # Giữ lại dấu gạch ngang của Spider-Man
    clean_str = re.sub(r'[^\w\s-]', '', raw_text.lower())
    words = clean_str.split()

    keywords = []
    for w in words:
        if w not in stop_words and len(w) > 1 and w not in keywords:
            keywords.append(w)

    return " ".join(keywords).title()

@app.post("/identify-by-image")
async def identify_by_image(file: UploadFile = File(...)):
    try:
        image_data = await file.read()

        # 1. BẮT CẢ 2 THẰNG OCR VÀ BLIP CÙNG LÀM VIỆC
        ocr_texts = reader.readtext(image_data, detail=0)
        ocr_result = " ".join(ocr_texts).strip()
        ocr_query = extract_keywords(ocr_result)

        raw_image = Image.open(io.BytesIO(image_data)).convert('RGB')
        inputs = processor(raw_image, return_tensors="pt").to(device)
        out = model.generate(**inputs)
        ai_caption = processor.decode(out[0], skip_special_tokens=True)
        blip_query = extract_keywords(ai_caption)

        print(f"👁️ OCR đọc được: '{ocr_query}'")
        print(f"🧠 BLIP tả cảnh: '{blip_query}'")

        # 2. LÊN DANH SÁCH TỪ KHÓA CẦN TÌM (Ưu tiên từ trên xuống dưới)
        queries_to_try = []

        if ocr_query:
            queries_to_try.append(ocr_query) # Lớp 1: Thử nguyên câu OCR

        if blip_query:
            queries_to_try.append(blip_query) # Lớp 2: Thử đoán cảnh của BLIP

        if ocr_query:
            words = ocr_query.split()
            # Lớp 3: Thử xóa từ đầu tiên (Vì font chữ Logo phim ở đầu thường bị đọc sai)
            if len(words) > 1:
                queries_to_try.append(" ".join(words[1:]))

            # Lớp 4: Nhặt ra 2 từ dài nhất trong ảnh để tìm (Từ dài thường ít bị sai chính tả)
            longest_words = sorted(words, key=len, reverse=True)
            for w in longest_words[:2]:
                if len(w) > 4:
                    queries_to_try.append(w)

        # Xóa các từ khóa trùng lặp
        queries_to_try = list(dict.fromkeys([q for q in queries_to_try if q]))
        print(f"🎯 Danh sách rải thảm: {queries_to_try}")

        # 3. MANG DANH SÁCH ĐI OANH TẠC TMDb
        async with httpx.AsyncClient() as client_http:
            search_url = f"{TMDB_BASE_URL}/search/multi"
            found_item = None

            for query in queries_to_try:
                print(f"🔍 Đang dò tìm: '{query}'")
                params = {"api_key": TMDB_API_KEY, "query": query, "language": "vi-VN"}
                response = await client_http.get(search_url, params=params)
                data = response.json()

                # Chỉ lấy Phim hoặc TV Show
                results = [item for item in data.get("results", []) if item.get("media_type") in ["movie", "tv"]]
                if results:
                    found_item = results[0]
                    print(f"✅ BÙM! Trúng đích với từ khóa: '{query}'")
                    break # Tìm thấy là thoát vòng lặp ngay!

        # 4. XỬ LÝ KẾT QUẢ TRẢ VỀ ANDROID
        if not found_item:
            return {"message": "AI bó tay, ảnh quá mờ hoặc chữ quá khó!"}

        title = found_item.get("title") or found_item.get("name")
        release_date = found_item.get("release_date") or found_item.get("first_air_date", "N/A")
        overview = found_item.get("overview", "Chưa có nội dung")
        poster_path = found_item.get("poster_path")

        movie_info = {
            "_id": str(len(search_history_db) + 1),
            "tmdb_id": found_item["id"],
            "title": title,
            "release_date": release_date,
            "overview": overview,
            "poster_path": f"https://image.tmdb.org/t/p/w500{poster_path}" if poster_path else None,
            "search_time": datetime.utcnow().isoformat()
        }
        search_history_db.append(movie_info)
        return {"id": movie_info["_id"], "message": "Đã lưu", "movie_info": movie_info}

    except Exception as e:
        return {"error": str(e)}

@app.get("/history")
def get_history():
    # Trả về 10 phim tìm kiếm gần nhất
    return {"search_history": list(reversed(search_history_db))[:10]}

ngrok.set_auth_token("38FjUsvfnOEV7wfrFhBl4jMGlxU_4Q4orQVBRbjkQqsU8GfYM")

public_url = ngrok.connect(8000).public_url
print("="*50)
print(f"🟢 LINK API CHO ANDROID: {public_url}/")
print("="*50)

# Khởi động Server tích hợp với vòng lặp của Colab
config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
await server.serve()

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


🚀 Đang chạy Server trên: CUDA
Đang nạp OCR...
Đang nạp BLIP...


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

✅ AI SẴN SÀNG!
🟢 LINK API CHO ANDROID: https://untentered-adalynn-wickedly.ngrok-free.dev/


INFO:     Started server process [6234]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


👁️ OCR đọc được: 'Spidermt Into Spider-Verse'
🧠 BLIP tả cảnh: 'Spiderman Into Spiderverse'
🎯 Danh sách rải thảm: ['Spidermt Into Spider-Verse', 'Spiderman Into Spiderverse', 'Into Spider-Verse', 'Spider-Verse', 'Spidermt']
🔍 Đang dò tìm: 'Spidermt Into Spider-Verse'
🔍 Đang dò tìm: 'Spiderman Into Spiderverse'
✅ BÙM! Trúng đích với từ khóa: 'Spiderman Into Spiderverse'
INFO:     42.112.255.186:0 - "POST /identify-by-image HTTP/1.1" 200 OK


/tmp/ipykernel_6234/870060878.py:248: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "search_time": datetime.utcnow().isoformat()


👁️ OCR đọc được: ''
🧠 BLIP tả cảnh: 'Spider Man'
🎯 Danh sách rải thảm: ['Spider Man']
🔍 Đang dò tìm: 'Spider Man'
✅ BÙM! Trúng đích với từ khóa: 'Spider Man'
INFO:     42.112.255.186:0 - "POST /identify-by-image HTTP/1.1" 200 OK
👁️ OCR đọc được: 'Juufu Kkais2I'
🧠 BLIP tả cảnh: 'Seven Knights'
🎯 Danh sách rải thảm: ['Juufu Kkais2I', 'Seven Knights', 'Kkais2I', 'Juufu']
🔍 Đang dò tìm: 'Juufu Kkais2I'
🔍 Đang dò tìm: 'Seven Knights'
✅ BÙM! Trúng đích với từ khóa: 'Seven Knights'
INFO:     42.112.255.186:0 - "POST /identify-by-image HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [6234]
